# Exercise 18.1: Reproducible Train / Validation / Test Split

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-18/01-train-val-test-split.ipynb)

**Reference:** Chapter 18 — Generalisation, Overfitting and Regularisation (`chapters/generalisation-and-regularisation.tex`)  
**Estimated Time:** 15–20 min  

### Learning Objectives
- Split a dataset into disjoint training, validation, and test subsets deterministically using `np.random.default_rng`.
- Standardise features using training-set mean and standard deviation only (zero data leakage).
- Verify disjointness, split proportions, and reproducible shuffling.


In [ ]:
import numpy as np


### Mathematical Background

A machine learning model is judged by how well it **generalises** to unseen data (Section 18.1). Standard practice partitions $N$ samples into three disjoint sets:
- **Training set** ($60\%$–$80\%$): used to fit model weights.
- **Validation set** ($10\%$–$20\%$): used to tune hyperparameters and trigger early stopping.
- **Test set** ($10\%$–$20\%$): held out until the very end for an unbiased evaluation.

Any feature scaling ($z = (x - \mu)/\sigma$) must fit $\mu$ and $\sigma$ **exclusively on the training set**.


In [ ]:
def train_val_test_split(
    X: np.ndarray,
    y: np.ndarray,
    train_ratio: float = 0.6,
    val_ratio: float = 0.2,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Deterministically split (X, y) into disjoint train, validation, and test sets.

    Args:
        X: Feature matrix of shape (n_samples, n_features).
        y: Target array of shape (n_samples,).
        train_ratio: Fraction of samples for training.
        val_ratio: Fraction of samples for validation.
        seed: Random seed for np.random.default_rng.

    Returns:
        Tuple (X_train, X_val, X_test, y_train, y_val, y_test).
    """
    # TODO: Use np.random.default_rng(seed).permutation(n) to slice X and y into 3 disjoint splits
    raise NotImplementedError("Complete this TODO")


def standardise_splits(
    X_train: np.ndarray,
    X_val: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Standardise train, val, and test splits using ONLY training-set statistics.

    Formula:
        mu = mean(X_train, axis=0)
        sigma = std(X_train, axis=0)
        X_scaled = (X - mu) / sigma

    Args:
        X_train: Training features.
        X_val: Validation features.
        X_test: Test features.

    Returns:
        Tuple (X_train_scaled, X_val_scaled, X_test_scaled).
    """
    # TODO: Fit mean and std on X_train only, then transform all three splits
    raise NotImplementedError("Complete this TODO")


<details>
<summary>Hint</summary>

- Generate a shuffled index array with `perm = np.random.default_rng(seed).permutation(len(X))`.
- Slice `perm` at `n_train = int(n * train_ratio)` and `n_train + int(n * val_ratio)` so the test set gets all remaining indices.
- In `standardise_splits`, compute `mu` and `sigma` from `X_train` only, then subtract `mu` and divide by `sigma` on all three arrays.

</details>


In [ ]:
rng = np.random.default_rng(42)
X = rng.normal(loc=10.0, scale=3.0, size=(100, 4))
# Encode sample index in y to verify disjointness
y = np.arange(100)

X_tr, X_va, X_te, y_tr, y_va, y_te = train_val_test_split(X, y, 0.6, 0.2, seed=42)
assert X_tr.shape == (60, 4) and X_va.shape == (20, 4) and X_te.shape == (20, 4)
all_ids = np.concatenate([y_tr, y_va, y_te])
assert len(set(all_ids.tolist())) == 100, 'Splits must be disjoint and cover all samples'

# Reproducibility check
_, _, _, y_tr2, _, _ = train_val_test_split(X, y, 0.6, 0.2, seed=42)
assert np.array_equal(y_tr, y_tr2)

# Leak-free standardisation check
X_tr_s, X_va_s, X_te_s = standardise_splits(X_tr, X_va, X_te)
assert np.allclose(X_tr_s.mean(axis=0), 0.0, atol=1e-9)
assert np.allclose(X_tr_s.std(axis=0), 1.0, atol=1e-9)
assert X_va_s.shape == X_va.shape and X_te_s.shape == X_te.shape

print('All Chapter 18.1 assertions passed!')


### Think Deeper

1. Why is it a mistake to tune hyperparameters (such as regularisation strength $\lambda$) on the test set?
2. For an imbalanced classification dataset (e.g., 1% fraud), why might you prefer a **stratified** split over a purely random permutation?
